# Time series generation ONLINE


In [ ]:
# ============================================================
# BPIC2017 and 2012 ONLINE DAILY SERIES: CUTOFF-AVAILABLE FEATURES
# Daily elapsed-time cohort: cases with >=1 event on that date.
# Actor features: transitions assigned to timestamp_j, then cumulatively summed.
# ============================================================
import pandas as pd, numpy as np, pm4py, zipfile, shutil
from pathlib import Path
from google.colab import files

print('STEP 1/2: upload the original BPIC2017 event log (.xes, .xes.gz, or ZIP)')
raw_upload = files.upload()
raw_files = [n for n in raw_upload if n.lower().endswith(('.xes', '.xes.gz', '.zip'))]
if len(raw_files) != 1: raise ValueError('Upload exactly one raw BPIC2017 XES/XES.GZ/ZIP file.')
raw_path = raw_files[0]
if raw_path.lower().endswith('.zip'):
    with zipfile.ZipFile(raw_path) as zf:
        members = [n for n in zf.namelist() if n.lower().endswith(('.xes', '.xes.gz'))]
        if len(members) != 1: raise ValueError(f'Expected one XES file in ZIP; found {members[:10]}')
        RAW_XES_PATH = str(Path('/content/bpic2017_raw') / Path(members[0]).name)
        Path(RAW_XES_PATH).parent.mkdir(parents=True, exist_ok=True)
        with zf.open(members[0]) as src, open(RAW_XES_PATH, 'wb') as dst: shutil.copyfileobj(src, dst)
else: RAW_XES_PATH = raw_path

log = pm4py.read_xes(RAW_XES_PATH)
df_eventlog = pm4py.convert_to_dataframe(log)
CASE_COL, ACTIVITY_COL, TIMESTAMP_COL, RESOURCE_COL = 'case:concept:name', 'concept:name', 'time:timestamp', 'org:resource'
need = [CASE_COL, ACTIVITY_COL, TIMESTAMP_COL, RESOURCE_COL]
missing = [c for c in need if c not in df_eventlog]
if missing: raise KeyError(f'Missing required BPIC2017 event columns: {missing}')
df_eventlog[TIMESTAMP_COL] = pd.to_datetime(df_eventlog[TIMESTAMP_COL], errors='coerce', utc=True).dt.tz_convert(None)
df_eventlog = df_eventlog.dropna(subset=[CASE_COL, TIMESTAMP_COL]).copy()
df_eventlog[CASE_COL] = df_eventlog[CASE_COL].astype(str)

print('STEP 2/2: upload the enriched BPIC2017 actor-behavior CSV')
ab_upload = files.upload()
ab_files = [n for n in ab_upload if n.lower().endswith('.csv')]
if len(ab_files) != 1: raise ValueError('Upload exactly one actor-behavior results CSV.')
ab = pd.read_csv(ab_files[0], dtype={'case_id': str})
need_ab = ['case_id', 'behavior', 'timestamp_j', 'delta_t']
missing = [c for c in need_ab if c not in ab]
if missing: raise KeyError(f'Missing columns in actor-behavior CSV: {missing}')
ab['timestamp_j'] = pd.to_datetime(ab['timestamp_j'], errors='coerce', utc=True).dt.tz_convert(None)
if not pd.api.types.is_numeric_dtype(ab['delta_t']):
    try: ab['delta_t'] = pd.to_timedelta(ab['delta_t']).dt.total_seconds()
    except (TypeError, ValueError): ab['delta_t'] = pd.to_numeric(ab['delta_t'], errors='coerce')
ab['delta_t'] = pd.to_numeric(ab['delta_t'], errors='coerce')
ab = ab.dropna(subset=need_ab).copy()
ab['case_id'] = ab['case_id'].astype(str)

# case_end is used only to define the future Avg_Remaining_Time outcome.
case_bounds = df_eventlog.groupby(CASE_COL)[TIMESTAMP_COL].agg(case_start='min', case_end='max')
behavior_order = ['C', 'I', 'HI', 'HB', 'HD']
source_behaviors = set(ab['behavior'].astype(str).str.strip())
behaviors = [b for b in behavior_order if b in source_behaviors]
behaviors += sorted(source_behaviors.difference(behavior_order))
days = pd.date_range(df_eventlog[TIMESTAMP_COL].min().normalize(), df_eventlog[TIMESTAMP_COL].max().normalize(), freq='D')

# A case belongs to date d's cohort iff it has an event timestamped on d.
# This depends only on events through that day, not on its eventual case_end.
df_eventlog['event_day'] = df_eventlog[TIMESTAMP_COL].dt.normalize()
case_day = df_eventlog[[CASE_COL, 'event_day']].drop_duplicates().merge(
    case_bounds.reset_index(), on=CASE_COL, how='left'
)
case_day['snapshot_ts'] = case_day['event_day'] + pd.Timedelta(days=1)
case_day['elapsed_hours'] = (case_day['snapshot_ts'] - case_day['case_start']).dt.total_seconds() / 3600.0
# Outcome only: completed cases in the day's cohort have zero remaining time at EOD.
case_day['remaining_hours'] = (case_day['case_end'] - case_day['snapshot_ts']).dt.total_seconds().div(3600.0).clip(lower=0)
daily_kpis = case_day.groupby('event_day').agg(
    Avg_Elapsed_Time=('elapsed_hours', 'mean'),
    Avg_Remaining_Time=('remaining_hours', 'mean'),
    Cases_With_Activity=(CASE_COL, 'nunique')
)

# Process-state measures available at the end-of-day prediction cutoff.
# Resource activity means a resource appears on at least one event that day.
daily_event_count = df_eventlog.groupby('event_day').size().reindex(days, fill_value=0)
resource_by_day = df_eventlog.groupby('event_day')[RESOURCE_COL].nunique().reindex(days, fill_value=0)
daily_kpis['Events_During_Day'] = daily_event_count.astype(int)
daily_kpis['Active_Resources_During_Day'] = resource_by_day.astype(int)

# Each transition contributes on the calendar day of timestamp_j. The exported
# Cum_* features are cumulative totals through each date, across all cases;
# no case-end or active-case filter is used.
ab['transition_day'] = ab['timestamp_j'].dt.normalize()
daily_count = ab.groupby(['transition_day', 'behavior']).size().unstack(fill_value=0)
daily_time = ab.groupby(['transition_day', 'behavior'])['delta_t'].sum().unstack(fill_value=0)
daily_count = daily_count.reindex(index=days, columns=behaviors, fill_value=0)
daily_time = daily_time.reindex(index=days, columns=behaviors, fill_value=0)
daily_kpis['Transitions_Completed_During_Day'] = (
    ab.groupby('transition_day').size().reindex(days, fill_value=0).astype(int)
)
df_behavior = pd.DataFrame(index=days)
for behavior in behaviors:
    df_behavior[f'Cum_Count_{behavior}'] = daily_count[behavior].cumsum()
    seconds = daily_time[behavior].cumsum()
    df_behavior[f'Cum_Time_{behavior}_seconds'] = seconds
    df_behavior[f'Cum_Time_{behavior}_minutes'] = seconds / 60.0
    df_behavior[f'Cum_Time_{behavior}_hours'] = seconds / 3600.0
    df_behavior[f'Cum_Time_{behavior}_days'] = seconds / 86400.0

# Keep every calendar date. Avg_Remaining_Time is a target; do not use it as input.
df_final_online = daily_kpis.reindex(days).join(df_behavior, how='left').sort_index()
df_final_online['Cases_With_Activity'] = df_final_online['Cases_With_Activity'].fillna(0).astype(int)
for col in df_behavior.columns:
    df_final_online[col] = df_final_online[col].fillna(0)
print('BPIC2017 daily series shape:', df_final_online.shape)
display(df_final_online.head())
OUTPUT_CSV = 'BPIC2017_final_online_daily_time_series_cutoff_safe.csv'
df_final_online.to_csv(OUTPUT_CSV, index_label='date')
files.download(OUTPUT_CSV)




  Welcome to PM4Py — Community Version
  Open-Source License (AGPL v3)

  📚 Docs & Examples:
     https://processintelligence.solutions/pm4py

  ⚖️  License: AGPL v3 — Commercial use requires open-sourcing your application.
     Business use without open-sourcing? A commercial license is available:
     https://processintelligence.solutions/pm4py#licensing




STEP 1/2: upload the original BPIC2017 event log (.xes, .xes.gz, or ZIP)


Saving BPI Challenge 2017.xes.zip to BPI Challenge 2017.xes.zip


/usr/local/lib/python3.13/dist-packages/pm4py/utils.py:1027: UserWarning: Install the optional requirement `r4pm` to import/export files faster. `rustxes` remains supported as a fallback.
  warnings.warn(


parsing log, completed traces ::   0%|          | 0/31509 [00:00<?, ?it/s]

STEP 2/2: upload the enriched BPIC2017 actor-behavior CSV


Saving BPIC2017_actor_behavior_results.csv to BPIC2017_actor_behavior_results.csv
BPIC2017 daily series shape: (398, 26)


,Avg_Elapsed_Time,Avg_Remaining_Time,Cases_With_Activity,Events_During_Day,Active_Resources_During_Day,Transitions_Completed_During_Day,Cum_Count_C,Cum_Time_C_seconds,Cum_Time_C_minutes,Cum_Time_C_hours,...,Cum_Count_HI,Cum_Time_HI_seconds,Cum_Time_HI_minutes,Cum_Time_HI_hours,Cum_Time_HI_days,Cum_Count_HD,Cum_Time_HD_seconds,Cum_Time_HD_minutes,Cum_Time_HD_hours,Cum_Time_HD_days
2016-01-01,8.315309,559.785954,22,126,1,104,104,1430.0,23.833333,0.397222,...,0,0.0,0.000000,0.000000,0.000000,0,0.0,0.000000,0.000000,0.000000
2016-01-02,16.836255,579.941254,65,554,6,508,574,40553.0,675.883333,11.264722,...,17,532812.0,8880.200000,148.003333,6.166806,19,1046800.0,17446.666667,290.777778,12.115741
2016-01-03,8.573110,617.757570,37,207,1,170,742,42740.0,712.333333,11.872222,...,17,532812.0,8880.200000,148.003333,6.166806,19,1046800.0,17446.666667,290.777778,12.115741
2016-01-04,25.316220,582.883255,203,2062,22,1941,2456,192900.0,3215.000000,53.583333,...,91,3136742.0,52279.033333,871.317222,36.304884,155,9503204.0,158386.733333,2639.778889,109.990787
2016-01-05,24.875910,546.435861,203,1757,16,1658,3908,389500.0,6491.666667,108.194444,...,141,4389473.0,73157.883333,1219.298056,50.804086,297,16842161.0,280702.683333,4678.378056,194.932419


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# BPIC2012 ONLINE DAILY SERIES: CUTOFF-AVAILABLE FEATURES
# Daily elapsed-time cohort: cases with >=1 event on that date.
# Actor features: transitions assigned to timestamp_j, then cumulatively summed.
# ============================================================
import pandas as pd, numpy as np, pm4py, zipfile, shutil
from pathlib import Path
from google.colab import files

print('STEP 1/2: upload the original BPIC2012 event log (.xes, .xes.gz, or ZIP)')
raw_upload = files.upload()
raw_files = [n for n in raw_upload if n.lower().endswith(('.xes', '.xes.gz', '.zip'))]
if len(raw_files) != 1: raise ValueError('Upload exactly one raw BPIC2012 XES/XES.GZ/ZIP file.')
raw_path = raw_files[0]
if raw_path.lower().endswith('.zip'):
    with zipfile.ZipFile(raw_path) as zf:
        members = [n for n in zf.namelist() if n.lower().endswith(('.xes', '.xes.gz'))]
        if len(members) != 1: raise ValueError(f'Expected one XES file in ZIP; found {members[:10]}')
        RAW_XES_PATH = str(Path('/content/bpic2012_raw') / Path(members[0]).name)
        Path(RAW_XES_PATH).parent.mkdir(parents=True, exist_ok=True)
        with zf.open(members[0]) as src, open(RAW_XES_PATH, 'wb') as dst: shutil.copyfileobj(src, dst)
else: RAW_XES_PATH = raw_path

log = pm4py.read_xes(RAW_XES_PATH)
df_eventlog = pm4py.convert_to_dataframe(log)
CASE_COL, ACTIVITY_COL, TIMESTAMP_COL, RESOURCE_COL = 'case:concept:name', 'concept:name', 'time:timestamp', 'org:resource'
need = [CASE_COL, ACTIVITY_COL, TIMESTAMP_COL, RESOURCE_COL]
missing = [c for c in need if c not in df_eventlog]
if missing: raise KeyError(f'Missing required BPIC2017 event columns: {missing}')
df_eventlog[TIMESTAMP_COL] = pd.to_datetime(df_eventlog[TIMESTAMP_COL], errors='coerce', utc=True).dt.tz_convert(None)
df_eventlog = df_eventlog.dropna(subset=[CASE_COL, TIMESTAMP_COL]).copy()
df_eventlog[CASE_COL] = df_eventlog[CASE_COL].astype(str)

print('STEP 2/2: upload the enriched BPIC2012 actor-behavior CSV')
ab_upload = files.upload()
ab_files = [n for n in ab_upload if n.lower().endswith('.csv')]
if len(ab_files) != 1: raise ValueError('Upload exactly one actor-behavior results CSV.')
ab = pd.read_csv(ab_files[0], dtype={'case_id': str})
need_ab = ['case_id', 'behavior', 'timestamp_j', 'delta_t']
missing = [c for c in need_ab if c not in ab]
if missing: raise KeyError(f'Missing columns in actor-behavior CSV: {missing}')
ab['timestamp_j'] = pd.to_datetime(ab['timestamp_j'], errors='coerce', utc=True).dt.tz_convert(None)
if not pd.api.types.is_numeric_dtype(ab['delta_t']):
    try: ab['delta_t'] = pd.to_timedelta(ab['delta_t']).dt.total_seconds()
    except (TypeError, ValueError): ab['delta_t'] = pd.to_numeric(ab['delta_t'], errors='coerce')
ab['delta_t'] = pd.to_numeric(ab['delta_t'], errors='coerce')
ab = ab.dropna(subset=need_ab).copy()
ab['case_id'] = ab['case_id'].astype(str)

# case_end is used only to define the future Avg_Remaining_Time outcome.
case_bounds = df_eventlog.groupby(CASE_COL)[TIMESTAMP_COL].agg(case_start='min', case_end='max')
behavior_order = ['C', 'I', 'HI', 'HB']
behaviors = [b for b in behavior_order if b in set(ab['behavior'].astype(str))]
days = pd.date_range(df_eventlog[TIMESTAMP_COL].min().normalize(), df_eventlog[TIMESTAMP_COL].max().normalize(), freq='D')

# A case belongs to date d's cohort iff it has an event timestamped on d.
# This depends only on events through that day, not on its eventual case_end.
df_eventlog['event_day'] = df_eventlog[TIMESTAMP_COL].dt.normalize()
case_day = df_eventlog[[CASE_COL, 'event_day']].drop_duplicates().merge(
    case_bounds.reset_index(), on=CASE_COL, how='left'
)
case_day['snapshot_ts'] = case_day['event_day'] + pd.Timedelta(days=1)
case_day['elapsed_hours'] = (case_day['snapshot_ts'] - case_day['case_start']).dt.total_seconds() / 3600.0
# Outcome only: completed cases in the day's cohort have zero remaining time at EOD.
case_day['remaining_hours'] = (case_day['case_end'] - case_day['snapshot_ts']).dt.total_seconds().div(3600.0).clip(lower=0)
daily_kpis = case_day.groupby('event_day').agg(
    Avg_Elapsed_Time=('elapsed_hours', 'mean'),
    Avg_Remaining_Time=('remaining_hours', 'mean'),
    Cases_With_Activity=(CASE_COL, 'nunique')
)

# Each transition contributes on the calendar day of timestamp_j. The exported
# Cum_* features are cumulative totals through each date, across all cases;
# no case-end or active-case filter is used.
ab['transition_day'] = ab['timestamp_j'].dt.normalize()
daily_count = ab.groupby(['transition_day', 'behavior']).size().unstack(fill_value=0)
daily_time = ab.groupby(['transition_day', 'behavior'])['delta_t'].sum().unstack(fill_value=0)
daily_count = daily_count.reindex(index=days, columns=behaviors, fill_value=0)
daily_time = daily_time.reindex(index=days, columns=behaviors, fill_value=0)
df_behavior = pd.DataFrame(index=days)
for behavior in behaviors:
    df_behavior[f'Cum_Count_{behavior}'] = daily_count[behavior].cumsum()
    seconds = daily_time[behavior].cumsum()
    df_behavior[f'Cum_Time_{behavior}_seconds'] = seconds
    df_behavior[f'Cum_Time_{behavior}_minutes'] = seconds / 60.0
    df_behavior[f'Cum_Time_{behavior}_hours'] = seconds / 3600.0
    df_behavior[f'Cum_Time_{behavior}_days'] = seconds / 86400.0

# Keep every calendar date. Avg_Remaining_Time is a target; do not use it as input.
df_final_online = daily_kpis.reindex(days).join(df_behavior, how='left').sort_index()
df_final_online['Cases_With_Activity'] = df_final_online['Cases_With_Activity'].fillna(0).astype(int)
for col in df_behavior.columns:
    df_final_online[col] = df_final_online[col].fillna(0)
print('BPIC2012 daily series shape:', df_final_online.shape)
display(df_final_online.head())
OUTPUT_CSV = 'BPIC2012_final_online_daily_time_series_cutoff_safe.csv'
df_final_online.to_csv(OUTPUT_CSV, index_label='date')
files.download(OUTPUT_CSV)




  Welcome to PM4Py — Community Version
  Open-Source License (AGPL v3)

  📚 Docs & Examples:
     https://processintelligence.solutions/pm4py

  ⚖️  License: AGPL v3 — Commercial use requires open-sourcing your application.
     Business use without open-sourcing? A commercial license is available:
     https://processintelligence.solutions/pm4py#licensing




STEP 1/2: upload the original BPIC2012 event log (.xes, .xes.gz, or ZIP)


Saving BPI_Challenge_2012.xes.gz to BPI_Challenge_2012.xes.gz


/usr/local/lib/python3.13/dist-packages/pm4py/utils.py:1027: UserWarning: Install the optional requirement `r4pm` to import/export files faster. `rustxes` remains supported as a fallback.
  warnings.warn(


parsing log, completed traces ::   0%|          | 0/13087 [00:00<?, ?it/s]

STEP 2/2: upload the enriched BPIC2012 actor-behavior CSV


Saving BPIC2012_actor_behavior_results.csv to BPIC2012_actor_behavior_results.csv
BPIC2012 daily series shape: (167, 23)


,Avg_Elapsed_Time,Avg_Remaining_Time,Cases_With_Activity,Cum_Count_C,Cum_Time_C_seconds,Cum_Time_C_minutes,Cum_Time_C_hours,Cum_Time_C_days,Cum_Count_I,Cum_Time_I_seconds,...,Cum_Count_HI,Cum_Time_HI_seconds,Cum_Time_HI_minutes,Cum_Time_HI_hours,Cum_Time_HI_days,Cum_Count_HB,Cum_Time_HB_seconds,Cum_Time_HB_minutes,Cum_Time_HB_hours,Cum_Time_HB_days
2011-09-30,1.354293,296.626952,1,0,0.000,0.000000,0.000000,0.000000,0,0.000,...,0,0.000,0.000000,0.000000,0.000000,0,0.000,0.000000,0.000000,0.000000
2011-10-01,10.940633,278.586317,45,255,25669.799,427.829983,7.130500,0.297104,5,10093.963,...,18,100915.457,1681.924283,28.032071,1.168003,18,92551.403,1542.523383,25.708723,1.071197
2011-10-02,10.146456,212.055758,48,367,27286.202,454.770033,7.579501,0.315813,7,10170.240,...,18,100915.457,1681.924283,28.032071,1.168003,18,92551.403,1542.523383,25.708723,1.071197
2011-10-03,20.526997,232.119517,163,1241,109022.080,1817.034667,30.283911,1.261830,26,49907.369,...,92,2491017.001,41516.950017,691.949167,28.831215,112,3355608.510,55926.808500,932.113475,38.838061
2011-10-04,22.001849,265.697105,135,1984,180480.602,3008.010033,50.133501,2.088896,44,153262.395,...,171,3773728.841,62895.480683,1048.258011,43.677417,209,6277359.559,104622.659317,1743.710989,72.654625


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# BPIC2011 ONLINE DAILY SERIES: CUTOFF-AVAILABLE FEATURES
# Avg_Elapsed_Time and Avg_Remaining_Time are in days.
# Actor durations are exported once, in seconds.
# ============================================================
import pandas as pd
import numpy as np
import pm4py
import zipfile
import shutil
from pathlib import Path
from google.colab import files

print("STEP 1/2: upload the original BPIC2011 event log (.xes, .xes.gz, or ZIP)")
raw_upload = files.upload()
raw_files = [
    n for n in raw_upload
    if n.lower().endswith((".xes", ".xes.gz", ".zip"))
]
if len(raw_files) != 1:
    raise ValueError("Upload exactly one raw BPIC2011 XES, XES.GZ, or ZIP file.")

raw_path = raw_files[0]
if raw_path.lower().endswith(".zip"):
    with zipfile.ZipFile(raw_path) as zf:
        members = [
            n for n in zf.namelist()
            if n.lower().endswith((".xes", ".xes.gz"))
        ]
        if len(members) != 1:
            raise ValueError(
                f"Expected one XES file in ZIP; found {members[:10]}"
            )
        RAW_XES_PATH = str(
            Path("/content/bpic2011_raw") / Path(members[0]).name
        )
        Path(RAW_XES_PATH).parent.mkdir(parents=True, exist_ok=True)
        with zf.open(members[0]) as src, open(RAW_XES_PATH, "wb") as dst:
            shutil.copyfileobj(src, dst)
else:
    RAW_XES_PATH = raw_path

log = pm4py.read_xes(RAW_XES_PATH)
df_eventlog = pm4py.convert_to_dataframe(log)

CASE_COL = "case:concept:name"
ACTIVITY_COL = "concept:name"
TIMESTAMP_COL = "time:timestamp"
RESOURCE_COL = "org:group"

need = [CASE_COL, ACTIVITY_COL, TIMESTAMP_COL, RESOURCE_COL]
missing = [c for c in need if c not in df_eventlog.columns]
if missing:
    raise KeyError(f"Missing required BPIC2011 event columns: {missing}")

LOCAL_TZ = "Europe/Amsterdam"
_raw_ts_utc = pd.to_datetime(
    df_eventlog[TIMESTAMP_COL], errors="coerce", utc=True
)
df_eventlog["_timestamp_utc"] = _raw_ts_utc.dt.tz_localize(None)
df_eventlog[TIMESTAMP_COL] = (
    _raw_ts_utc.dt.tz_convert(LOCAL_TZ).dt.tz_localize(None)
)
df_eventlog = df_eventlog.dropna(
    subset=[CASE_COL, TIMESTAMP_COL, "_timestamp_utc"]
).copy()
df_eventlog[CASE_COL] = df_eventlog[CASE_COL].astype(str)

print("STEP 2/2: upload the enriched BPIC2011 actor-behavior CSV")
ab_upload = files.upload()
ab_files = [n for n in ab_upload if n.lower().endswith(".csv")]
if len(ab_files) != 1:
    raise ValueError("Upload exactly one actor-behavior results CSV.")

ab = pd.read_csv(ab_files[0], dtype={"case_id": str})
need_ab = ["case_id", "behavior", "timestamp_j", "delta_t"]
missing = [c for c in need_ab if c not in ab.columns]
if missing:
    raise KeyError(f"Missing columns in actor-behavior CSV: {missing}")

ab["timestamp_j"] = (
    pd.to_datetime(ab["timestamp_j"], errors="coerce", utc=True)
    .dt.tz_convert(LOCAL_TZ)
    .dt.tz_localize(None)
)
if not pd.api.types.is_numeric_dtype(ab["delta_t"]):
    try:
        ab["delta_t"] = pd.to_timedelta(ab["delta_t"]).dt.total_seconds()
    except (TypeError, ValueError):
        ab["delta_t"] = pd.to_numeric(ab["delta_t"], errors="coerce")

# Numeric delta_t values are assumed to be seconds.
ab["delta_t"] = pd.to_numeric(ab["delta_t"], errors="coerce")
ab = ab.dropna(subset=need_ab).copy()
ab["case_id"] = ab["case_id"].astype(str)
ab["behavior"] = ab["behavior"].astype(str).str.strip()

# Case end is used only to define the future Avg_Remaining_Time outcome.
case_bounds = df_eventlog.groupby(CASE_COL)["_timestamp_utc"].agg(
    case_start_utc="min",
    case_end_utc="max",
)

behavior_order = ["C", "I", "HI", "HB", "HD"]
source_behaviors = set(ab["behavior"])
behaviors = [b for b in behavior_order if b in source_behaviors]
behaviors += sorted(source_behaviors.difference(behavior_order))

days = pd.date_range(
    df_eventlog[TIMESTAMP_COL].min().normalize(),
    df_eventlog[TIMESTAMP_COL].max().normalize(),
    freq="D",
)

# A case is in the daily cohort if it has an event on that date.
df_eventlog["event_day"] = df_eventlog[TIMESTAMP_COL].dt.normalize()
case_day = (
    df_eventlog[[CASE_COL, "event_day"]]
    .drop_duplicates()
    .merge(case_bounds.reset_index(), on=CASE_COL, how="left")
)

# End-of-day cutoff: local midnight at the start of the next calendar day.
_local_cutoffs = (
    pd.DatetimeIndex(case_day["event_day"]).tz_localize(LOCAL_TZ)
    + pd.DateOffset(days=1)
)
case_day["snapshot_utc"] = (
    _local_cutoffs.tz_convert("UTC").tz_localize(None).to_numpy()
)

# KPI targets are durations in days.
case_day["elapsed_days"] = (
    case_day["snapshot_utc"] - case_day["case_start_utc"]
).dt.total_seconds() / 86400.0

case_day["remaining_days"] = (
    (case_day["case_end_utc"] - case_day["snapshot_utc"])
    .dt.total_seconds()
    .div(86400.0)
    .clip(lower=0)
)

daily_kpis = case_day.groupby("event_day").agg(
    Avg_Elapsed_Time=("elapsed_days", "mean"),
    Avg_Remaining_Time=("remaining_days", "mean"),
    Cases_With_Activity=(CASE_COL, "nunique"),
).reindex(days)

# Process-state measures available at the end-of-day prediction cutoff.
daily_event_count = (
    df_eventlog.groupby("event_day")
    .size()
    .reindex(days, fill_value=0)
)
resource_by_day = (
    df_eventlog.groupby("event_day")[RESOURCE_COL]
    .nunique()
    .reindex(days, fill_value=0)
)
daily_kpis["Events_During_Day"] = daily_event_count.astype(int)
daily_kpis["Active_Resources_During_Day"] = resource_by_day.astype(int)

# Actor transitions are assigned to the day of timestamp_j.
ab["transition_day"] = ab["timestamp_j"].dt.normalize()
daily_count = (
    ab.groupby(["transition_day", "behavior"])
    .size()
    .unstack(fill_value=0)
)
daily_time_seconds = (
    ab.groupby(["transition_day", "behavior"])["delta_t"]
    .sum()
    .unstack(fill_value=0)
)
daily_count = daily_count.reindex(
    index=days, columns=behaviors, fill_value=0
)
daily_time_seconds = daily_time_seconds.reindex(
    index=days, columns=behaviors, fill_value=0
)

daily_kpis["Transitions_Completed_During_Day"] = (
    ab.groupby("transition_day")
    .size()
    .reindex(days, fill_value=0)
    .astype(int)
)

df_behavior = pd.DataFrame(index=days)
for behavior in behaviors:
    df_behavior[f"Cum_Count_{behavior}"] = daily_count[behavior].cumsum()

    # Keep only one duration scale to avoid perfectly collinear predictors.
    df_behavior[f"Cum_Time_{behavior}_seconds"] = (
        daily_time_seconds[behavior].cumsum()
    )

# Keep every calendar date. Remaining time is an outcome, not an input.
df_final_online = (
    daily_kpis.reindex(days)
    .join(df_behavior, how="left")
    .sort_index()
)
df_final_online["Cases_With_Activity"] = (
    df_final_online["Cases_With_Activity"].fillna(0).astype(int)
)
for col in df_behavior.columns:
    df_final_online[col] = df_final_online[col].fillna(0)

print("BPIC2011 daily series shape:", df_final_online.shape)
display(df_final_online.head())

OUTPUT_CSV = (
    "BPIC2011_final_online_daily_time_series_cutoff_safe_days.csv"
)
df_final_online.to_csv(OUTPUT_CSV, index_label="date")
files.download(OUTPUT_CSV)



  Welcome to PM4Py — Community Version
  Open-Source License (AGPL v3)

  📚 Docs & Examples:
     https://processintelligence.solutions/pm4py

  ⚖️  License: AGPL v3 — Commercial use requires open-sourcing your application.
     Business use without open-sourcing? A commercial license is available:
     https://processintelligence.solutions/pm4py#licensing




STEP 1/2: upload the original BPIC2011 event log (.xes, .xes.gz, or ZIP)


Saving Hospital_log.xes.gz to Hospital_log.xes.gz


/usr/local/lib/python3.13/dist-packages/pm4py/utils.py:1027: UserWarning: Install the optional requirement `r4pm` to import/export files faster. `rustxes` remains supported as a fallback.
  warnings.warn(


parsing log, completed traces ::   0%|          | 0/1143 [00:00<?, ?it/s]

STEP 2/2: upload the enriched BPIC2011 actor-behavior CSV


Saving BPIC2011_actor_behavior_results.csv to BPIC2011_actor_behavior_results.csv
BPIC2011 daily series shape: (1173, 14)


,Avg_Elapsed_Time,Avg_Remaining_Time,Cases_With_Activity,Events_During_Day,Active_Resources_During_Day,Transitions_Completed_During_Day,Cum_Count_C,Cum_Time_C_seconds,Cum_Count_I,Cum_Time_I_seconds,Cum_Count_HI,Cum_Time_HI_seconds,Cum_Count_HB,Cum_Time_HB_seconds
2005-01-03,1.000000,603.400000,5,44,6,39,33,0.0,0,0.0,6,0.0,0,0.0
2005-01-04,NaN,NaN,0,0,0,0,33,0.0,0,0.0,6,0.0,0,0.0
2005-01-05,3.000000,444.000000,2,5,3,5,34,0.0,0,0.0,10,345600.0,0,0.0
2005-01-06,1.750000,576.750000,4,67,5,64,86,0.0,0,0.0,21,345600.0,1,259200.0
2005-01-07,1.666667,652.666667,3,7,3,6,87,0.0,0,0.0,26,518400.0,1,259200.0


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>